# Import Libraries

In [1]:

!git clone https://github.com/fcdalgic/GramBeddings.git
%cd /content/GramBeddings
!ls

Cloning into 'GramBeddings'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 120 (delta 23), reused 33 (delta 9), pack-reused 69 (from 1)
Receiving objects: 100% (120/120), 113.03 MiB | 14.24 MiB/s, done.
Resolving deltas: 100% (37/37), done.
Updating files: 100% (54/54), done.
/content/GramBeddings
adverserial_generator.py  Model.py		       train.bat
data			  NGramSequenceTransformer.py  train.py
feature_extraction.py	  README.md		       Utils
index.html		  single_channel_train.py
LICENSE			  test.py


In [4]:
import os
import pathlib
import glob
import getpass
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
from pathlib import Path
import sys, time

In [6]:
# Clone original source and check available data

repo = pathlib.Path('/content/GramBeddings')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/fcdalgic/GramBeddings.git',str(repo)],check=True)
for folder in ['/content/grambeddings_data', '/content']:
    p = pathlib.Path(folder)
    print(folder, [str(x) for x in p.glob('**/train.csv')][:10], [str(x) for x in p.glob('**/test.csv')][:10])
import tensorflow as tf
print('TensorFlow:',tf.__version__,'GPU:',tf.config.list_physical_devices('GPU'))

/content/grambeddings_data [] []
/content ['/content/GramBeddings/data/ISCXURL2016/train.csv', '/content/GramBeddings/data/PhishStorm/train.csv', '/content/GramBeddings/data/PDRCNN/train.csv', '/content/GramBeddings/data/kaggle/03-siddharthkumar25/train.csv', '/content/GramBeddings/data/kaggle/01-sid321axn/train.csv', '/content/GramBeddings/data/kaggle/02-akshaya1508/train.csv'] ['/content/GramBeddings/data/ISCXURL2016/test.csv', '/content/GramBeddings/data/PhishStorm/test.csv', '/content/GramBeddings/data/PDRCNN/test.csv', '/content/GramBeddings/data/kaggle/03-siddharthkumar25/test.csv', '/content/GramBeddings/data/kaggle/01-sid321axn/test.csv', '/content/GramBeddings/data/kaggle/02-akshaya1508/test.csv']
TensorFlow: 2.21.0 GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Data Preprocessing

In [10]:

DATA_DIR = Path("/content/grambeddings_data")

# Read CSV files: label first, URL second
train_df = pd.read_csv(
    DATA_DIR / "train.csv",
    header=None,
    names=["label", "url"],
    sep=",",
    dtype={"label": str, "url": str},
    on_bad_lines="skip"
)

test_df = pd.read_csv(
    DATA_DIR / "test.csv",
    header=None,
    names=["label", "url"],
    sep=",",
    dtype={"label": str, "url": str},
    on_bad_lines="skip"
)

# Convert labels: 1 = phishing, 2 = legitimate
def prepare_data(df):
    df = df.copy()
    df["label"] = df["label"].str.strip()

    df = df[df["label"].isin(["1", "2"])]
    df["label"] = df["label"].map({
        "1": 1,
        "2": 0
    })

    df = df.dropna(subset=["label", "url"])
    df = df[df["url"].str.len() >= 8]

    return df.reset_index(drop=True)

train_df = prepare_data(train_df)
test_df = prepare_data(test_df)

print("Train rows:", len(train_df))
print("Train labels:", train_df["label"].value_counts().to_dict())

print("Test rows:", len(test_df))
print("Test labels:", test_df["label"].value_counts().to_dict())

# Small balanced subset for a quick first experiment
FAST_RUN = True
TRAIN_PER_CLASS = 10000
TEST_PER_CLASS = 2500

if FAST_RUN:
    train_df = (
        train_df.groupby("label", group_keys=False)
        .sample(n=TRAIN_PER_CLASS, random_state=42)
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    test_df = (
        test_df.groupby("label", group_keys=False)
        .sample(n=TEST_PER_CLASS, random_state=42)
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

# Save data for the training script.
# Original loader expects label 1 = phishing, -1 = legitimate,
# with tab-separated values.
OUTPUT_DIR = Path("/content/GramBeddings/data/grambeddings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, data in [("train", train_df), ("test", test_df)]:
    output = data.copy()
    output["label"] = output["label"].map({1: 1, 0: -1})
    output.to_csv(
        OUTPUT_DIR / f"{name}.csv",
        sep="\t",
        header=False,
        index=False
    )

print("\nPrepared", len(train_df), "train rows")
print("Prepared", len(test_df), "validation rows")
print("Files saved to:", OUTPUT_DIR)


Train rows: 636803
Train labels: {0: 319631, 1: 317172}
Test rows: 159188
Test labels: {0: 79892, 1: 79296}

Prepared 20000 train rows
Prepared 5000 validation rows
Files saved to: /content/GramBeddings/data/grambeddings


In [11]:
# Minimal compatibility fixes

transformer_file = repo/'NGramSequenceTransformer.py'
s = transformer_file.read_text()
s = s.replace('vectorizer.get_feature_names()', 'vectorizer.get_feature_names_out()')
# Keras Embedding must reserve index zero for padding; the original n-gram IDs start at 1.
s = s.replace('vocab_size = len(self.tk.word_index)\n', 'vocab_size = len(self.tk.word_index) + 1\n')
transformer_file.write_text(s)

model_file = repo/'Model.py'
s = model_file.read_text()
# GlobalMaxPooling1D removes the time dimension; a second Conv1D needs it.
# Local pooling retains a sequence while preserving the CNN -> CNN -> BiLSTM -> attention design.
s = s.replace('GlobalMaxPooling1D(data_format = "channels_first")(x)', 'MaxPooling1D(pool_size=2)(x)')
model_file.write_text(s)

(repo/'outputs/training/models').mkdir(parents=True,exist_ok=True)
(repo/'outputs/tensorboard').mkdir(parents=True,exist_ok=True)
print('Patched scikit-learn feature names, n-gram vocabulary size and invalid pooling operation')

Patched scikit-learn feature names, n-gram vocabulary size and invalid pooling operation


# Training

In [12]:
# Train the four-branch model with the authors' training script

args = [sys.executable, 'train.py', '--dataset', 'grambeddings',
        '--model_name', 'grambeddings_colab',
        '--ngram_1','3','--ngram_2','4','--ngram_3','5',
        '--max_seq_len','128','--attn_width','10',
        '--embed_dim','15','--max_features','160000',
        '--max_df','0.7','--min_df','0.000001',
        '--rnn_cell_size','256','--batch_size','128', '--epochs','3']
start = time.time()

# Stream training output directly to Colab
process = subprocess.Popen(
    args,
    cwd=repo,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"}
)

for line in process.stdout:
    print(line, end='', flush=True)

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")

print("\nTraining completed!")
print("Training elapsed minutes:", round((time.time() - start) / 60, 1))

Training elapsed minutes: 2.1


# Results

In [13]:
# Export metrics. Original authors' results.csv lacks a header; its schema comes from SaveResults.

results_path=repo/'outputs/training/results.csv'
assert results_path.exists(), 'Training did not create results.csv'
cols = ['dataset','ngram_1','ngram_2','ngram_3','max_features','max_df','min_df',
        'max_seq_len','embed_dim','attention_width','rnn_cell_size','batch_size',
        'warm_start','warm_mode','epochs','best_epoch','case_insensitive',
        'train_accuracy','validation_accuracy','validation_precision',
        'validation_recall','validation_auc','training_seconds']
results=pd.read_csv(results_path,header=None,names=cols)
last=results.tail(1).copy()
p=last['validation_precision'].astype(float)
r=last['validation_recall'].astype(float)
last['validation_f1']=2*p*r/(p+r).replace(0,float('nan'))
last['train_examples']=len(train_df)
last['validation_examples']=len(test_df)
last['validation_is_used_for_checkpoint_selection']=True
last['source_repo']='https://github.com/fcdalgic/GramBeddings'
last['run_type']='subset' if FAST_RUN else 'full dataset'
output='/content/GramBeddings_metrics.csv'
last.to_csv(output,index=False)
display(last[['run_type','train_examples','validation_examples','best_epoch',
              'validation_accuracy','validation_precision','validation_recall',
              'validation_f1','validation_auc']])
print('Saved:',output)
print('Saved model(s):',list((repo/'outputs/training/models').glob('*')))


,run_type,train_examples,validation_examples,best_epoch,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_auc
0,subset,20000,5000,2,0.9466,0.965791,0.926,0.945477,0.989267


Saved: /content/GramBeddings_metrics.csv
Saved model(s): [PosixPath('/content/GramBeddings/outputs/training/models/grambeddings_colab.h5')]
